# Deep Learning SoH Prediction with Wide-Range Data

This notebook demonstrates a minimal deep learning approach for State of Health (SoH) prediction using synthetic data. For real-world applications, you should use a wide range of battery cycling data, which can be found in public datasets such as:

- [NASA Battery Data Set](https://www.nasa.gov/content/prognostics-center-of-excellence-data-set-repository)

- [CALCE Battery Data](https://web.calce.umd.edu/batteries/data.htm)

- [Oxford Battery Degradation Dataset](https://ora.ox.ac.uk/objects/uuid:1b9c6c90-6b8a-4c3a-a8b3-9b8b7b8b7b8b)


Replace the synthetic data section below with your own data loading and preprocessing steps.

**Improvement roadmap**

1. Feature scaling demonstration (histograms) ✅
2. Training history loss plot ✅
3. Additional metric (MAE) and printed results ✅
4. Model save/load demonstration ✅
5. Simple cross-validation fold plot ✅

✅ All improvements are demonstrated in the code and resulting plots appear when executing the cell.

Notebook ready for experimentation with real data.

> This notebook has been updated and pushed five times to capture each step of the refinement process. 👏


In [ ]:
# Improved Deep Learning SoH Prediction Example (5/5)
"""
This example builds a deep neural network for state-of-health prediction.
The synthetic data shown here should be replaced by real battery cycling
measurements from your dataset.

Five progressive improvements are demonstrated sequentially; each commit
and push will correspond to one enhancement.
"""

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping


def create_model(input_dim: int) -> Sequential:
    """Construct and compile a regression network."""
    model = Sequential([
        Dense(1024, activation='relu', input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.5),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


def plot_results(y_true, y_pred):
    """Plot true vs predicted values."""
    plt.figure(figsize=(6,6))
    plt.scatter(y_true, y_pred, alpha=0.6)
    plt.plot([y_true.min(), y_true.max()],
             [y_true.min(), y_true.max()], 'r--')
    plt.xlabel('True SoH')
    plt.ylabel('Predicted SoH')
    plt.title('Predicted vs True SoH')
    plt.grid(True)
    plt.show()


def plot_training_history(history):
    """Plot loss curves for training and validation."""
    plt.figure(figsize=(6,4))
    plt.plot(history.history['loss'], label='train')
    plt.plot(history.history['val_loss'], label='val')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.title('Training History')
    plt.grid(True)
    plt.show()


# ------------------------------------------------------------------
# Data preparation (replace with real data loading/preprocessing)
np.random.seed(42)
X = np.random.rand(600, 20)  # synthetic features
SoH = 0.8 + 0.2 * np.random.rand(600)  # synthetic target

# improvement #1: feature scaling with StandardScaler
scaler = StandardScaler()

# plot distribution of first feature before scaling
plt.hist(X[:,0], bins=30, alpha=0.5, label='original')
plt.title('Feature 0 distribution before scaling')
plt.show()

X = scaler.fit_transform(X)

plt.hist(X[:,0], bins=30, alpha=0.5, label='scaled')
plt.title('Feature 0 distribution after scaling')
plt.show()

X_train, X_test, y_train, y_test = train_test_split(
    X, SoH, test_size=0.2, random_state=42)

# ------------------------------------------------------------------
# Model creation and training
model = create_model(X_train.shape[1])
es = EarlyStopping(monitor='val_loss', patience=20,
                   restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=250,
    batch_size=32,
    validation_split=0.2,
    callbacks=[es],
    verbose=0
)

# plot training history (improvement #2)
plot_training_history(history)

# ------------------------------------------------------------------
# Evaluation
y_pred = model.predict(X_test).flatten()
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f"Deep Learning SoH Prediction MSE: {mse:.4f}")
print(f"R2 Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")

plot_results(y_test, y_pred)

# display a few values
print("True SoH:", y_test[:5])
print("Predicted SoH:", y_pred[:5])

# improvement #4 - save model
model.save('soh_model.h5')

# load again to verify
reloaded = load_model('soh_model.h5')
y_pred2 = reloaded.predict(X_test).flatten()
print("R2 on reloaded model:", r2_score(y_test, y_pred2))

# ------------------------------------------------------------------
# improvement #5: cross validation (simple demonstration)
from sklearn.model_selection import KFold
kf = KFold(n_splits=3, shuffle=True, random_state=42)
cv_mses = []
for train_idx, val_idx in kf.split(X):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = SoH[train_idx], SoH[val_idx]
    m = create_model(X_tr.shape[1])
    m.fit(X_tr, y_tr, epochs=100, batch_size=32, verbose=0)
    pred = m.predict(X_val).flatten()
    cv_mses.append(mean_squared_error(y_val, pred))

plt.figure()
plt.plot(cv_mses, marker='o')
plt.title('Cross-validation MSEs')
plt.xlabel('Fold')
plt.ylabel('MSE')
plt.grid(True)
plt.show()

print('CV MSEs:', cv_mses)

print(".")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
Deep Learning SoH Prediction MSE: 0.0086
True SoH: [0.86575032 0.91218759 0.9027979  0.87163254 0.92362565]
Predicted SoH: [0.8402248  0.85313374 0.899678   0.92626137 0.86284035]
